In [1]:
import numpy as np
import pandas as pd
import cartopy.crs as ccrs
import geopandas as gpd
from shapely.geometry import Polygon, MultiPolygon, Point
from shapely.ops import transform
import matplotlib.colors as colors
import matplotlib.pyplot as plt
import xarray as xr
from glob import glob
from windspharm.xarray import VectorWind

In [2]:
# read in basin definition file
polygons_dict = {}

# read in basin definition file
with open("tc_basins.dat", "r") as f:
    for line in f:
        line = line.strip()
        if not line or line.startswith("#"):
            continue

        parts = line.split(",")
        basin_name = parts[0].replace('"', '')
        n_vertices = int(parts[1])

        lon_vals = list(map(float, parts[2:2+n_vertices]))
        lat_vals = list(map(float, parts[2+n_vertices:2+2*n_vertices]))

        coords = list(zip(lon_vals, lat_vals))
        poly = Polygon(coords)

        if basin_name not in polygons_dict:
            polygons_dict[basin_name] = []
        polygons_dict[basin_name].append(poly)

# Convert to GeoDataFrame
basin_records = []

for name, poly_list in polygons_dict.items():
    if len(poly_list) == 1:
        geom = poly_list[0]
    else:
        geom = MultiPolygon(poly_list)

    basin_records.append({
        "basin name": name,
        "geometry": geom
    })

basins = gpd.GeoDataFrame(basin_records, crs="EPSG:4326")

# fix invalid polygons
basins["geometry"] = basins["geometry"].buffer(0)

# remove empy geometries
basins = basins[~basins.geometry.is_empty]

# convert basins' lon to -180-180
basins["geometry"] = basins["geometry"].apply(
    lambda geom: transform(
        lambda x, y: (((x + 180) % 360) - 180, y),
        geom
    )
)

# read in NAtl subbasin polygons
sub_polygons_dict = {}

with open("tc_subbasins_NAtl_v5.dat", "r") as f:
    for line in f:
        line = line.strip()
        if not line or line.startswith("#"):
            continue

        parts = line.split(",")
        sub_basin_name = parts[0].replace('"', '')
        n_vertices = int(parts[1])

        lon_vals = list(map(float, parts[2:2+n_vertices]))
        lon_vals = [(lon + 180) % 360 - 180 for lon in lon_vals]
        lat_vals = list(map(float, parts[2+n_vertices:2+2*n_vertices]))

        coords = list(zip(lon_vals, lat_vals))
        poly = Polygon(coords)

        if sub_basin_name not in sub_polygons_dict:
            sub_polygons_dict[sub_basin_name] = []
        sub_polygons_dict[sub_basin_name].append(poly)

# Convert to GeoDataFrame
sub_basin_records = []

for name, poly_list in sub_polygons_dict.items():
    if len(poly_list) == 1:
        geom = poly_list[0]
    else:
        geom = MultiPolygon(poly_list)

    sub_basin_records.append({
        "sub_basin_name": name,
        "geometry": geom
    })

sub_basins = gpd.GeoDataFrame(sub_basin_records, crs="EPSG:4326",geometry="geometry")

# fix invalid polygons
sub_basins["geometry"] = sub_basins["geometry"].buffer(0)

# remove empty geometries
sub_basins = sub_basins[~sub_basins.geometry.is_empty]

# longitude conversion
import shapely.ops
def shift_lon(geom):
    return shapely.ops.transform(
        lambda x, y: (((x + 180) % 360) - 180, y),
        geom
    )

# shift lon
sub_basins["geometry"] = sub_basins["geometry"].apply(shift_lon)

In [ ]:
# # load in u and v wind data to calculate 850hPa vorticity
# ds1 = xr.open_mfdataset(
#     "datasets/u-wind/*.nc", 
#     combine = "by_coords", 
#     preprocess=lambda ds: ds.drop_vars("time_bnds", errors="ignore")
# )
# ds2 = xr.open_mfdataset(
#     "datasets/v-wind/*.nc", 
#     combine = "by_coords", 
#     preprocess=lambda ds: ds.drop_vars("time_bnds", errors="ignore")
# )

# # # print(ds1)
# # # print(ds2)

# # select the variables
# uwnd = ds1["uwnd"]
# vwnd = ds2["vwnd"]

# # convert lon to -180-180
# uwnd = uwnd.assign_coords(
#     lon=(((uwnd.lon + 180) % 360) - 180)
# ).sortby("lon")
# vwnd = vwnd.assign_coords(
#     lon=(((vwnd.lon + 180) % 360) - 180)
# ).sortby("lon")

# # filter to relative humidity to a specific pressure level only
# uwnd850 = uwnd.sel(level=850)
# vwnd850 = vwnd.sel(level=850)

# # filter to hurricane season
# uwnd850_szn = (
#     uwnd850
#     .where(uwnd850.time.dt.month.isin([6, 7, 8, 9, 10]), drop=True)
# )
# vwnd850_szn = (
#     vwnd850
#     .where(vwnd850.time.dt.month.isin([6, 7, 8, 9, 10]), drop=True)
# )

# # compute daily relative vorticity
# w = VectorWind(uwnd850_szn, vwnd850_szn)
# vort850 = w.vorticity()

# # compute planetary vorticity and add to relative (to calc absolute vort)
# omega = 7.2921e-5
# f = 2 * omega * np.sin(np.deg2rad(vort850.lat))
# abs_vort = vort850 + f

# # filter to N Atlantic basin
# abs_vort = abs_vort.rio.write_crs("EPSG:4326")
# region = basins[basins["basin name"] == "N Atlantic"]
# abs_vort = abs_vort.rio.clip(region.geometry, region.crs, drop=True)



# a_abs = np.abs(1e5 * abs_vort) ** 1.5
# a_clip = np.maximum(1e5 * abs_vort, 0) ** 1.5
# print("max difference:",
#       (a_abs - a_clip).max(skipna=True).item())

# print("mean difference:",
#       (a_abs - a_clip).mean(skipna=True).item())





# # # average to monthly
# # abs_vort_monthly = abs_vort.resample(time="1MS").mean()

# print(abs_vort)

# # # save dataset
# abs_vort.to_netcdf("datasets/GPI/GPI_EN_calc/abs_vort_850_daily.nc")
